# cNMF Evaluation Pipeline Tutorial

This notebook demonstrates how to evaluate gene programs discovered by cNMF.

**Prerequisites**: You should have already run cNMF inference (prepare, factorize, combine, consensus)
and saved results as `.h5mu` MuData files (see the inference tutorial).

## Pipeline Steps

| Step | Description |
|------|-------------|
| **Step 1. Set Up** | Import libraries, define I/O paths (`out_dir`, `run_name`), resource file paths (GWAS, normalized counts), MuData keys, and the (K, threshold) grid to evaluate |
| **Step 2. Load Non-targeting Guides (Optional)** | Read guide annotation file and extract non-targeting guide target labels as the reference set for perturbation tests |
| **Step 3. Load Resources for Explained Variance** | Create the cNMF object and load the normalized expression matrix for explained variance calculation |
| **Step 4. Load Guide Data** | Load the MuData/AnnData containing guide assignment information and define a helper to attach guide metadata to each MuData |
| **Step 5. Evaluation Loop** | For each (K, threshold) combination, load the `.h5mu`, attach guide metadata, then run all enabled evaluation sub-steps (5a–5g) |

### Evaluation Sub-steps (inside Step 5)

| Sub-step | Output File | Description |
|----------|-------------|-------------|
| **5a. Categorical Association** | `{K}_categorical_association_results.txt`, `{K}_categorical_association_posthoc.txt` | Kruskal-Wallis test + Dunn's posthoc to test whether program scores differ across categories (e.g. batch, condition) |
| **5b. Reactome Enrichment** | `{K}_geneset_enrichment_reactome.txt` | Fisher exact test of top 300 program genes against Reactome pathways |
| **5c. GO Enrichment** | `{K}_GO_term_enrichment.txt` | Fisher exact test of top 300 program genes against GO Biological Process terms |
| **5d. Trait Enrichment** | `{K}_trait_enrichment.txt` | Fisher exact test of top program genes against GWAS-linked genes from Open Targets L2G |
| **5e. Perturbation Association** | `{K}_perturbation_association_{sample}.txt` | Mann-Whitney U test per sample: do perturbed cells have shifted program scores vs non-targeting controls? |
| **5f. Explained Variance** | `{K}_explained_variance.tsv`, `{K}_explained_variance_summary.tsv` | Measure how much normalized expression variance each consensus program captures |
| **5g. Motif Enrichment** (optional) | `cNMF_{class}_pearson_topn2000_{sample}_motif_*.txt` | Pearson correlation of motif counts vs program gene scores for enhancer/promoter regions |

## Expected Output Structure

After running the evaluation pipeline, the output directory will have the following structure:

```
{out_dir}/{run_name}/
├── Inference/                                           # (from inference step — read-only inputs)
│   ├── cnmf_tmp/
│   │   └── Inference.norm_counts.h5ad                   # Used for explained variance
│   └── adata/
│       └── cNMF_{K}_{thresh}.h5mu                       # Input MuData per (K, threshold)
│
└── Evaluation/
    └── {K}_{thresh}/                                    # One folder per (K, threshold) combo
        ├── {K}_categorical_association_results.txt       # 5a. Kruskal-Wallis test per program
        ├── {K}_categorical_association_posthoc.txt       # 5a. Pairwise posthoc (Dunn) tests
        ├── {K}_geneset_enrichment_reactome.txt           # 5b. Reactome pathway enrichment (Fisher)
        ├── {K}_GO_term_enrichment.txt                    # 5c. GO Biological Process enrichment (Fisher)
        ├── {K}_trait_enrichment.txt                      # 5d. GWAS trait enrichment (Open Targets)
        ├── {K}_perturbation_association_{sample}.txt     # 5e. Per-sample perturbation U-test
        ├── {K}_explained_variance.tsv                    # 5f. Variance explained per program
        ├── {K}_explained_variance_summary.tsv            # 5f. Summary statistics
        └── (optional) motif enrichment files:            # 5g.
            ├── cNMF_enhancer_pearson_topn2000_{sample}_motif_match.txt
            ├── cNMF_enhancer_pearson_topn2000_{sample}_motif_count.txt
            ├── cNMF_enhancer_pearson_topn2000_{sample}_motif_enrichment.txt
            ├── cNMF_promoter_pearson_topn2000_{sample}_motif_match.txt
            ├── cNMF_promoter_pearson_topn2000_{sample}_motif_count.txt
            └── cNMF_promoter_pearson_topn2000_{sample}_motif_enrichment.txt
```

### Key Files

| File | Description |
|------|-------------|
| `{K}_categorical_association_results.txt` | One row per program with Kruskal-Wallis p-value testing whether program scores differ across the categorical variable |
| `{K}_categorical_association_posthoc.txt` | Pairwise posthoc comparisons (Dunn's test) between each pair of category levels |
| `{K}_geneset_enrichment_reactome.txt` | Fisher exact test results for top 300 program genes against Reactome 2022 pathways |
| `{K}_GO_term_enrichment.txt` | Fisher exact test results for top 300 program genes against GO Biological Process 2023 terms |
| `{K}_trait_enrichment.txt` | Fisher exact test for top program genes against GWAS-linked genes from Open Targets L2G |
| `{K}_perturbation_association_{sample}.txt` | Mann-Whitney U test per sample — one file per unique value in `categorical_key` |
| `{K}_explained_variance.tsv` | Per-program explained variance scores |

### Notes

- `{K}` = number of components (e.g., 30, 50, 100). `{thresh}` = density threshold with `.` replaced by `_` (e.g., `0_4`, `2_0`).
- `{sample}` corresponds to the unique values of the `categorical_key` column (e.g., condition labels).
- Perturbation association requires guide assignment data loaded from `mdata_guide_path`.
- Explained variance requires the normalized counts matrix from `Inference/cnmf_tmp/Inference.norm_counts.h5ad`.
- Motif enrichment is optional and requires additional resources (HOCOMOCO motifs, genome FASTA, scE2G loci files).
- These evaluation outputs are the input to the downstream **Interpretation** and **Calibration** pipelines.

In [ ]:
import os
import sys
import yaml
import logging
import mudata as mu
import pandas as pd
import cnmf
import scanpy as sc

# Change path to wherever you have repo locally
sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src')

from Evaluation.src import (
    compute_categorical_association,
    compute_geneset_enrichment,
    compute_trait_enrichment,
    compute_perturbation_association,
    compute_explained_variance,
    compute_motif_enrichment)

In [ ]:
## Step 1. Set Up

In [ ]:
# ── cNMF output paths ──
out_dir = "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result"          # directory passed as output_dir to cNMF
run_name = "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50"                       # name passed to cNMF

# ── Resource files ──
# Normalized counts produced by cNMF prepare (inside Inference/cnmf_tmp/)
X_normalized_path = f"{out_dir}/{run_name}/Inference/cnmf_tmp/Inference.norm_counts.h5ad"

# Open Targets GWAS data for trait enrichment
gwas_data_path = "/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Evaluation/Resources/OpenTargets_L2G_Filtered.csv.gz"

# (Optional) Guide annotation file — set to None if not using perturbation data
guide_annotation_path = None  # e.g. "/path/to/guide_metadata.tsv"

# ── Perturbation data (only needed for perturbation association) ──
# Path to the MuData or AnnData that contains guide assignment info
mdata_guide_path = None  # e.g. "/path/to/perturb_seq_data.h5ad"

In [ ]:
# ── MuData keys ──
# These must match the keys used when building the MuData in the inference step
data_key = "rna"              # key for the RNA expression modality
prog_key = "cNMF"             # key for the cNMF program modality
categorical_key = "batch" # obs column with sample/condition labels
organism = "human"            # "human" or "mouse"

# Guide-related keys (only needed for perturbation association)
guide_assignment_key = "guide_assignment"
guide_names_key = "guide_names"
guide_targets_key = "guide_targets"

# ── K and threshold grid to evaluate ──
components = [30, 50, 60, 80, 100, 200]              # values of K that were run in inference
sel_threshs = [0.4, 0.8, 2.0]       # density thresholds used in consensus

## Step 2. Load Non-targeting Guides (Optional)

In [4]:
# list of non-targeting guides
df_target = pd.read_csv(guide_annotation_path, sep = "\t", index_col = 0)
df_target_non = df_target[df_target["targeting"] == False]
reference_targets = df_target_non.index.values.tolist()

In [ ]:
## Step 3. Load Resources for Explained Variance

In [3]:
# objects used in explained variance calculation
cnmf_obj = cnmf.cNMF(output_dir=out_dir, name=run_name)
X_norm = sc.read_h5ad(X_normalized_path)
X = X_norm.X

In [ ]:
## Step 4. Load Guide Data

In [ ]:
# reformate the data for eval pipeline
def _assign_guide(mdata, ad_guide, gene_names_key='symbol'):
    mdata['cNMF'].uns['guide_targets'] = ad_guide.uns['guide_targets']
    mdata['cNMF'].obsm['guide_assignment'] = ad_guide.obsm['guide_assignment']
    mdata['cNMF'].uns['var_names'] = ad_guide.var[gene_names_key]
    mdata['rna'].var_names = ad_guide.var[gene_names_key].astype(str)
    mdata['rna'].var['var_names'] = ad_guide.var[gene_names_key].astype(str)

ad_guide = mu.read(mdata_guide_path)

## Step 5. Evaluation Loop

In [ ]:
for sel_thresh in sel_threshs:
    thresh_str = str(sel_thresh).replace(".", "_")

    for k in components:
        print(f"\n{'='*60}")
        print(f"Evaluating K={k}, density_threshold={sel_thresh}")
        print(f"{'='*60}")

        output_folder = f"{out_dir}/{run_name}/Evaluation/{k}_{thresh_str}"
        os.makedirs(output_folder, exist_ok=True)

        # ── Load MuData ──
        mdata_path = f"{out_dir}/{run_name}/Inference/adata/cNMF_{k}_{thresh_str}.h5mu"
        mdata = mu.read(mdata_path)

        # Attach guide metadata if available
        if ad_guide is not None:
            _assign_guide(mdata, ad_guide)

        # ────────────────────────────────────────────────────────────
        # 5a. Categorical Association
        #     Tests whether program scores differ across conditions
        #     (Kruskal-Wallis + posthoc pairwise tests)
        # ────────────────────────────────────────────────────────────
        print("Running categorical association...")
        results_df, posthoc_df = compute_categorical_association(
            mdata,
            prog_key=prog_key,
            categorical_key=categorical_key,
            pseudobulk_key=None,
            test="dunn",
            n_jobs=-1,
            inplace=False,
        )
        results_df.to_csv(f"{output_folder}/{k}_categorical_association_results.txt", sep="\t", index=False)
        posthoc_df.to_csv(f"{output_folder}/{k}_categorical_association_posthoc.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 5b. Gene-Set Enrichment (Reactome)
        #     Fisher test of top 300 program genes against Reactome pathways
        # ────────────────────────────────────────────────────────────
        print("Running Reactome gene-set enrichment...")
        reactome_res = compute_geneset_enrichment(
            mdata,
            prog_key=prog_key,
            data_key=data_key,
            organism=organism,
            library="Reactome_2022",
            method="fisher",
            database="enrichr",
            loading_rank_thresh=300,
            n_jobs=-1,
            inplace=False,
            use_loadings_gene=True,
        )
        reactome_res.to_csv(f"{output_folder}/{k}_geneset_enrichment_reactome.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 5c. Gene-Set Enrichment (GO Biological Process)
        #     Same as above but against GO terms
        # ────────────────────────────────────────────────────────────
        print("Running GO Biological Process enrichment...")
        go_res = compute_geneset_enrichment(
            mdata,
            prog_key=prog_key,
            data_key=data_key,
            organism=organism,
            library="GO_Biological_Process_2023",
            method="fisher",
            database="enrichr",
            loading_rank_thresh=300,
            n_jobs=-1,
            inplace=False,
            use_loadings_gene=True,
        )
        go_res.to_csv(f"{output_folder}/{k}_GO_term_enrichment.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 5d. Trait Enrichment (GWAS)
        #     Fisher test of top program genes against Open Targets
        #     GWAS locus-to-gene mappings
        # ────────────────────────────────────────────────────────────
        print("Running GWAS trait enrichment...")
        trait_res = compute_trait_enrichment(
            mdata,
            gwas_data=gwas_data_path,
            prog_key=prog_key,
            data_key=data_key,
            library="OT_GWAS",
            n_jobs=-1,
            inplace=False,
            key_column="trait_efos",
            gene_column="gene_name",
            method="fisher",
            loading_rank_thresh=300,
            use_loadings_gene=False,
        )
        trait_res.to_csv(f"{output_folder}/{k}_trait_enrichment.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 5e. Perturbation Association (requires guide data)
        #     Mann-Whitney U test: do perturbed cells have shifted
        #     program scores vs non-targeting controls?
        #     Runs separately per condition/sample.
        # ────────────────────────────────────────────────────────────
        print("Running perturbation association...")
        for samp in mdata["rna"].obs[categorical_key].unique():
            mdata_sub = mdata[mdata["rna"].obs[categorical_key] == samp]
            test_stats_df = compute_perturbation_association(
                mdata_sub,
                prog_key=prog_key,
                collapse_targets=True,
                pseudobulk=False,
                reference_targets=reference_targets,
                FDR_method="StoreyQ",
                n_jobs=-1,
                inplace=False,
            )
            test_stats_df.to_csv(
                f"{output_folder}/{k}_perturbation_association_{samp}.txt", sep="\t", index=False
                )

        # ────────────────────────────────────────────────────────────
        # 5f. Explained Variance
        #     How much of the normalized expression variance is
        #     captured by the consensus W and H matrices?
        # ────────────────────────────────────────────────────────────
        print("Computing explained variance...")
        compute_explained_variance(
            cnmf_obj,
            X,
            k,
            output_folder=output_folder,
            thre=str(sel_thresh),
            program_name=mdata[prog_key].var_names,
        )

        '''
        # ────────────────────────────────────────────────────────────
        # 5g. Motif Enrichment (Optional)
        #     Pearson correlation of motif counts vs program gene
        #     scores for enhancer/promoter regions using HOCOMOCO
        #     motifs and scE2G loci.
        # ────────────────────────────────────────────────────────────
        # Run motif enrichment
        fimo_thresh_enhancer = 1e-6
        fimo_thresh_promoter = 1e-4

        for samp in mdata['rna'].obs[categorical_key].unique():
            for class_, thresh in [('enhancer', fimo_thresh_enhancer), 
                                ('promoter', fimo_thresh_promoter)]:

                loci_file = '/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Evaluation/Resources/scE2G_links/EnhancerPredictionsAllPutative.ForVariantOverlap.shrunk150bp_D{}_{}.tsv'.format(i, class_)
                motif_match_df, motif_count_df, motif_enrichment_df = compute_motif_enrichment(
                    mdata, 
                    prog_key='cNMF',
                    data_key='rna',
                    motif_file='/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Evaluation/Resources/hocomoco_meme.meme',
                    seq_file='/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Evaluation/Resources/hg38.fa',
                    loci_file=loci_file,
                    window=1000,
                    sig=thresh,
                    eps=1e-4,
                    n_top=2000,
                    n_jobs=-1,
                    inplace=False,
                    gene_names_key=gene_names_key
                )

                motif_match_df.to_csv(os.path.join(out_dir, f'cNMF_{class_}_pearson_topn2000_{samp}_motif_match.txt'), sep='\t', index=False)
                motif_count_df.to_csv(os.path.join(out_dir, f'cNMF_{class_}_pearson_topn2000_{samp}_motif_count.txt'), sep='\t', index=False)
                motif_enrichment_df.to_csv(os.path.join(out_dir, f'cNMF_{class_}_pearson_topn2000_{samp}_motif_enrichment.txt'), sep='\t', index=False)
            '''

        print(f"Done. Results saved to {output_folder}")

## Motif Enrichment (Optional)

In [ ]:
# # Format files

# Thresholds
# score_thresh_abc_e2g_enhancer = 0.015
# score_thresh_abc_e2g_promoter = 0.8

# for i in range(4):
#     e2g = pd.read_csv('scE2G_links/EnhancerPredictionsAllPutative.ForVariantOverlap.shrunk150bp_D{}.tsv'.format(i), sep='\t')

#     e2g_enhancers = e2g.loc[(e2g['class']!='promoter') &\
#                             (e2g['ABC.Score']>score_thresh_abc_e2g_enhancer)]
#     e2g_enhancers = e2g_enhancers.loc[:,['chr', 'start', 'end', 'name', 'class', 'ABC.Score', 'TargetGene']]
#     e2g_enhancers.columns = [        'chromosome', 'start', 'end', 'seq_name', 'seq_class', 'seq_score', 'gene_name']

#     e2g_enhancers.to_csv('scE2G_links/EnhancerPredictionsAllPutative.ForVariantOverlap.shrunk150bp_D{}_enhancer.tsv'.format(i), 
#                           sep='\t', index=False)

#     e2g_promoters = e2g.loc[(e2g['class']=='promoter') &\
#                             (e2g['ABC.Score']>score_thresh_abc_e2g_promoter)]
#     e2g_promoters = e2g_promoters.loc[:,['chr', 'start', 'end', 'name', 'class', 'ABC.Score', 'TargetGene']]
#     e2g_promoters.columns = ['chromosome', 'start', 'end', 'seq_name', 'seq_class', 'seq_score', 'gene_name']

#     e2g_promoters.to_csv('scE2G_links/EnhancerPredictionsAllPutative.ForVariantOverlap.shrunk150bp_D{}_promoter.tsv'.format(i), 
#                         sep='\t', index=False)